# Team Rating Projection — Roster Counterfactual Tool

Interactive companion to `src/portalpoint/modeling/team_rating_projection.py` /
`scripts/run_team_rating_projection.py`, matching the module+script+notebook-in-sync
pattern used by all other models in this repo.

All fit/score/write logic lives in the `team_rating_projection` module — this notebook adds:
coverage audits, CV fold plots, coefficient inspection, slot-baseline heat maps, and
one-school counterfactual spot-checks that the script itself doesn't expose.

Full design/contract: `docs/models/team_rating_projection_plan.md`.
Production rerun: `scripts/run_team_rating_projection.py` (owns MLflow tracking +
promotion gate).

**Model version:** `team-roster-proj-v1`

**Prerequisites before running this notebook:**
1. `scripts/run_player_projection.py --phase cross-season` → fills Phase 2a neutral rows
2. `scripts/run_playing_time.py --target-season 2027` → fills `playing_time_projections`
3. `scripts/run_gap_matching.py` → fills `roster_baseline_members` for target season

Without step 2, Cell 8 will abort at the hard gate and Cells 9-11 will be empty.

In [ ]:
# Cell 0 — Imports + Config
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from portalpoint.modeling import team_rating_projection as trp
from portalpoint.modeling.io import get_sync_engine
from portalpoint.modeling.team_rating_projection import (
    MODEL_VERSION,
    ROSTER_FEATURES,
    NEUTRAL_MODEL_PRIORITY,
    PLAYING_TIME_MODEL_VERSION,
    CONFERENCE_TIER_CUTS,
)

TARGET_SEASON = 2027
SOURCE_SEASON = 2026
TRAIN_SEASONS = list(range(2021, 2027))   # 2021-2026 — all seasons with BartTorvik labels

# Rolling-origin CV folds (same structure as Player Projection / Destination Projection)
CV_FOLDS = [
    (list(range(2021, 2024)), 2024),   # Fold 1: train 2021-23, val 2024
    (list(range(2021, 2025)), 2025),   # Fold 2: train 2021-24, val 2025
    (list(range(2021, 2026)), 2026),   # Fold 3: train 2021-25, val 2026 (gate metric)
]

engine = get_sync_engine()
print(f"Model version : {MODEL_VERSION}")
print(f"Target season : {TARGET_SEASON}  Source season: {SOURCE_SEASON}")
print(f"Train seasons : {TRAIN_SEASONS}")
print(f"CV folds      : {[(tr[0], tr[-1], va) for tr, va in CV_FOLDS]}")
print(f"Neutral model priority: {NEUTRAL_MODEL_PRIORITY[:2]}")

## 1. Coverage Audit

Confirms that all upstream tables are populated before running the rest of the notebook.
- BartTorvik `team_season_stats` (adj_o/adj_d) is the training label source — check row count.
- `roster_baseline_members` populates baseline rosters; must have `season=TARGET_SEASON` rows.
- `playing_time_projections` is the hard gate for Cell 8 inference.

In [ ]:
# Cell 1 — Coverage audit
from sqlalchemy import text

with engine.connect() as conn:
    # BartTorvik team labels (training target)
    bart_counts = conn.execute(text("""
        SELECT season, count(*) AS n_teams,
               count(adj_o) AS n_with_adj_o, count(adj_d) AS n_with_adj_d
        FROM team_season_stats
        WHERE season = ANY(:seasons)
        GROUP BY season ORDER BY season
    """), {"seasons": TRAIN_SEASONS}).fetchall()

    # HE player RAPM (input feature, not label)
    rapm_counts = conn.execute(text("""
        SELECT season, count(*) AS n_players, count(off_adj_rapm) AS n_with_rapm
        FROM hoop_explorer_player_stats
        WHERE season = ANY(:seasons)
        GROUP BY season ORDER BY season
    """), {"seasons": TRAIN_SEASONS}).fetchall()

    # Roster baseline members
    rbm_counts = conn.execute(text("""
        SELECT season, count(*) AS n_members, count(DISTINCT school_id) AS n_schools
        FROM roster_baseline_members
        GROUP BY season ORDER BY season
    """)).fetchall()

    # Playing time projections (hard gate)
    pt_counts = conn.execute(text("""
        SELECT model_version, season, count(*) AS n_rows,
               count(DISTINCT player_id) AS n_players,
               count(DISTINCT school_id) AS n_schools
        FROM playing_time_projections
        WHERE season = :target_season
        GROUP BY 1, 2 ORDER BY 1
    """), {"target_season": TARGET_SEASON}).fetchall()

    # Team rating projections already written
    trp_counts = conn.execute(text("""
        SELECT model_version, season, count(*) AS n_rows,
               count(DISTINCT player_id) AS n_players,
               count(DISTINCT school_id) AS n_schools
        FROM team_rating_projections
        GROUP BY 1, 2 ORDER BY 2 DESC, 1
    """)).fetchall()

    # Portal candidates
    portal_count = conn.execute(text("""
        SELECT count(DISTINCT player_id)
        FROM player_team_fit_scores
        WHERE season = :season AND is_portal_candidate = true
    """), {"season": SOURCE_SEASON}).scalar()

print("--- BartTorvik team labels (training target) ---")
for season, n_teams, n_o, n_d in bart_counts:
    print(f"  season={season}  n_teams={n_teams:,}  adj_o={n_o:,}  adj_d={n_d:,}")

print("\n--- HE player RAPM (input feature, not label) ---")
for season, n_players, n_rapm in rapm_counts:
    print(f"  season={season}  n_players={n_players:,}  with_rapm={n_rapm:,} ({n_rapm/max(n_players,1):.0%})")

print("\n--- Roster baseline members ---")
for season, n_members, n_schools in rbm_counts:
    flag = " <- TARGET" if season == TARGET_SEASON else ""
    print(f"  season={season}  n_members={n_members:,}  n_schools={n_schools:,}{flag}")
if not any(r[0] == TARGET_SEASON for r in rbm_counts):
    print(f"  WARNING: no roster_baseline_members for season={TARGET_SEASON}")
    print(f"  Run: uv run python scripts/run_gap_matching.py first")

print(f"\n--- Playing time projections (target_season={TARGET_SEASON}, hard gate) ---")
if pt_counts:
    for version, season, nr, np_, ns in pt_counts:
        flag = " <- matches PLAYING_TIME_MODEL_VERSION" if version == PLAYING_TIME_MODEL_VERSION else ""
        print(f"  {version:30s}  rows={nr:,}  players={np_:,}  schools={ns:,}{flag}")
else:
    print(f"  EMPTY — run scripts/run_playing_time.py --target-season {TARGET_SEASON} first")

print(f"\n--- team_rating_projections already written ---")
if trp_counts:
    for version, season, nr, np_, ns in trp_counts:
        print(f"  {version:30s}  season={season}  rows={nr:,}  players={np_:,}  schools={ns:,}")
else:
    print("  None yet — will be populated by Cell 11")

print(f"\nPortal candidates (is_portal_candidate=True, season={SOURCE_SEASON}): {portal_count:,}")

## 2. Build Historical Roster States

Constructs the training feature matrix: one row per `(school_id, season)` in 2021-2026
with the 14-element ROSTER_FEATURES vector derived from actual player stats + HE RAPM.
BartTorvik `adj_o` / `adj_d` labels are joined at the end.

Also builds `slot_baselines` (average RAPM/shooting/usage by conference_tier × position),
used everywhere open rotation spots need filling.

In [ ]:
# Cell 2 — Build historical roster states (training data)
print(f"Building historical roster states for seasons {TRAIN_SEASONS}...")
features_df, labels_df, slot_baselines = trp.build_historical_roster_states(engine, TRAIN_SEASONS)

print(f"\nTraining data shape: {features_df.shape}")
print(f"  school-seasons: {len(features_df):,}")
print(f"  seasons in data: {sorted(features_df['season'].unique().tolist())}")
print(f"  schools per season:")
for s, g in features_df.groupby('season'):
    print(f"    {s}: {len(g):,}")

print(f"\nLabel stats (BartTorvik adj_o / adj_d):")
print(labels_df[["adj_o", "adj_d", "adj_em"]].describe().round(2))

print(f"\nSlot baselines: {len(slot_baselines)} (tier, position) keys")
sb_preview = {f"{k[0]}_{k[1]}": round(v['off_adj_rapm'], 3) for k, v in list(slot_baselines.items())[:6]}
print(f"  Sample off_adj_rapm by (tier_pos): {sb_preview}")

## 3. Slot Baseline Inspection

Slot baselines define the "typical replacement" quality for each conference tier × position.
Counterfactuals compare candidate vs. these baselines, not vs. zero — so their values are
load-bearing for the magnitude of every delta_adjEM output.

In [ ]:
# Cell 3 — Slot baseline inspection
if slot_baselines:
    sb_rows = []
    for (tier, pos), vals in sorted(slot_baselines.items()):
        sb_rows.append({"tier": tier, "position": pos, **{k: round(v, 3) for k, v in vals.items()}})
    sb_df = pd.DataFrame(sb_rows)

    print("Slot baselines — off_adj_rapm by tier × position:")
    pivot = sb_df.pivot_table(index="tier", columns="position", values="off_adj_rapm")
    display(pivot.round(3))

    print("\nSlot baselines — def_adj_rapm by tier × position:")
    pivot_def = sb_df.pivot_table(index="tier", columns="position", values="def_adj_rapm")
    display(pivot_def.round(3))

    # Heat map: off RAPM
    positions = sorted(sb_df["position"].unique())
    tiers = sorted(sb_df["tier"].unique())
    off_grid = np.full((len(tiers), len(positions)), np.nan)
    for _, row in sb_df.iterrows():
        ti = tiers.index(row["tier"])
        pi = positions.index(row["position"])
        off_grid[ti, pi] = row["off_adj_rapm"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    for ax, (grid, title) in zip(axes, [
        (off_grid, "off_adj_rapm (slot baseline)"),
        (np.full((len(tiers), len(positions)), np.nan), "def_adj_rapm (slot baseline)"),
    ]):
        if title.startswith("def"):
            for _, row in sb_df.iterrows():
                ti = tiers.index(row["tier"])
                pi = positions.index(row["position"])
                grid[ti, pi] = row["def_adj_rapm"]
        im = ax.imshow(grid, cmap="RdYlGn", aspect="auto")
        ax.set_xticks(range(len(positions)))
        ax.set_xticklabels(positions)
        ax.set_yticks(range(len(tiers)))
        ax.set_yticklabels([f"Tier {t}" for t in tiers])
        for ti in range(len(tiers)):
            for pi in range(len(positions)):
                val = grid[ti, pi]
                if not np.isnan(val):
                    ax.text(pi, ti, f"{val:.2f}", ha="center", va="center", fontsize=8)
        plt.colorbar(im, ax=ax)
        ax.set_title(title)
    plt.tight_layout()
    plt.show()
else:
    print("No slot baselines built — check that features_df is non-empty")

## 4. Feature Matrix EDA

Distribution inspection for each of the 14 ROSTER_FEATURES — confirms the feature
construction is working and identifies any extreme values or coverage gaps before fitting.

In [ ]:
# Cell 4 — Feature matrix EDA
print("Feature matrix summary stats:")
display(features_df[ROSTER_FEATURES].describe().round(3))

# Feature distributions — 3x5 grid
fig, axes = plt.subplots(3, 5, figsize=(18, 9))
axes_flat = axes.flatten()
for i, feat in enumerate(ROSTER_FEATURES):
    ax = axes_flat[i]
    features_df[feat].plot(kind="hist", bins=30, ax=ax, title=feat, fontsize=7)
    ax.set_xlabel("")
for j in range(len(ROSTER_FEATURES), len(axes_flat)):
    axes_flat[j].set_visible(False)
plt.suptitle("ROSTER_FEATURES distributions (2021-2026 training set)", y=1.01)
plt.tight_layout()
plt.show()

# Feature-label correlation
corr_df = features_df[ROSTER_FEATURES].copy()
corr_df["adj_em"] = labels_df["adj_em"].values
corr_with_em = corr_df.corr()["adj_em"].drop("adj_em").sort_values(ascending=False)
print("\nSpearman correlation with adj_em (sorted):")
display(corr_df.rank().corr()["adj_em"].drop("adj_em").sort_values(ascending=False).round(3).to_frame())

## 5. 3-Fold Rolling-Origin Cross-Validation

Validates predictive accuracy before fitting the final model. Fold 3 (val=2026) is the
gate metric for `maybe_promote`. Same CV structure as Player Projection Phase 2a and
Destination Projection.

In [ ]:
# Cell 5 — 3-fold rolling-origin CV
print("Running 3-fold rolling-origin cross-validation...")
cv_results = trp.rolling_origin_cv(features_df, labels_df, folds=CV_FOLDS)

print("\n--- CV results ---")
fold_rows = []
for fm in cv_results["fold_metrics"]:
    print(f"  Fold {fm['fold']} (val={fm['val_season']}):"
          f"  n_train={fm['n_train']:,}  n_val={fm['n_val']:,}")
    print(f"    off_rmse={fm['off_rmse']:.3f}  def_rmse={fm['def_rmse']:.3f}"
          f"  em_rmse={fm['em_rmse']:.3f}")
    print(f"    off_r2={fm['off_r2']:.3f}    def_r2={fm['def_r2']:.3f}")
    fold_rows.append(fm)

print(f"\nGate metric — fold3_em_rmse: {cv_results.get('fold3_em_rmse', 'N/A'):.3f}")
print(f"Mean em_rmse across folds  : {cv_results.get('mean_em_rmse', 'N/A'):.3f}")

# Bar charts: RMSE per fold
if fold_rows:
    fold_df = pd.DataFrame(fold_rows)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    fold_df.plot(
        x="val_season", y=["off_rmse", "def_rmse", "em_rmse"],
        kind="bar", ax=axes[0], rot=0, title="RMSE per fold"
    )
    axes[0].set_xlabel("Validation season")
    axes[0].set_ylabel("RMSE")

    fold_df.plot(
        x="val_season", y=["off_r2", "def_r2"],
        kind="bar", ax=axes[1], rot=0, title="R² per fold"
    )
    axes[1].set_xlabel("Validation season")
    axes[1].set_ylabel("R²")
    axes[1].axhline(0, color="black", linewidth=0.8)

    plt.tight_layout()
    plt.show()

## 6. Final Model Fit + Coefficient Report

Trains the final Ridge offense and defense models on all 2021-2026 data. Coefficient
inspection shows which roster features drive adj_o vs. adj_d — a sanity check that the
model has learned expected basketball relationships.

In [ ]:
# Cell 6 — Final model fit on all 2021-2026 training data
print("Fitting final team translation models on all training data...")
models = trp.fit_team_translation(features_df, labels_df)
models.slot_baselines = slot_baselines
models.train_seasons = TRAIN_SEASONS
models.cv_metrics = cv_results

print(f"  n_train_rows : {models.n_train_rows:,}")
print(f"  off_resid_std: {models.off_resid_std:.3f}")
print(f"  def_resid_std: {models.def_resid_std:.3f}")

# Coefficient report
coef_df = pd.DataFrame({
    "feature":   ROSTER_FEATURES,
    "off_coef":  models.off_model.coef_,
    "def_coef":  models.def_model.coef_,
}).set_index("feature")
coef_df["em_coef"] = coef_df["off_coef"] - coef_df["def_coef"]
coef_df = coef_df.sort_values("em_coef", key=abs, ascending=False)

print("\nModel coefficients (sorted by |em_coef|):")
display(coef_df.round(4))

# Visual: coefficient magnitudes
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(coef_df))
width = 0.35
ax.bar(x - width/2, coef_df["off_coef"], width, label="off_coef", color="#3498db", alpha=0.8)
ax.bar(x + width/2, coef_df["def_coef"], width, label="def_coef", color="#e74c3c", alpha=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(coef_df.index, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Coefficient (standardized scale)")
ax.set_title("Ridge coefficients: offense vs. defense models")
ax.legend()
plt.tight_layout()
plt.show()

## 7. MLflow Tracking

Logs params, CV metrics, model coefficients, and slot baselines to MLflow.
Checks promotion gate (`fold3_em_rmse` vs. `@champion`).

Production writes belong in `scripts/run_team_rating_projection.py` — this cell exists
as a one-run interactive equivalent. Set `SHOULD_LOG_MLFLOW = True` to execute.

In [ ]:
# Cell 7 — MLflow tracking (gated)
SHOULD_LOG_MLFLOW = False   # flip to True to run

EXPERIMENT_NAME  = "team-rating-projection"
MLFLOW_MODEL_NAME = "team-rating-scorer"
ARTIFACT_PATH    = "team_rating_models"
GATE_METRIC      = "fold3_em_rmse"

if SHOULD_LOG_MLFLOW:
    import pickle
    import tempfile
    import os
    import mlflow
    from portalpoint.modeling.mlflow_helpers import setup_mlflow, maybe_promote

    client = setup_mlflow(EXPERIMENT_NAME)

    with mlflow.start_run(run_name=f"team-roster-proj-s{TARGET_SEASON}") as run:
        # Params
        mlflow.log_params({
            "model_version":      MODEL_VERSION,
            "target_season":      TARGET_SEASON,
            "source_season":      SOURCE_SEASON,
            "train_seasons":      json.dumps(TRAIN_SEASONS),
            "neutral_model":      NEUTRAL_MODEL_PRIORITY[0],
            "playing_time_model": PLAYING_TIME_MODEL_VERSION,
            "n_train_rows":       models.n_train_rows,
            "n_features":         len(ROSTER_FEATURES),
        })

        # CV metrics
        for fm in cv_results.get("fold_metrics", []):
            mlflow.log_metrics({
                f"fold{fm['fold']}_off_rmse": fm["off_rmse"],
                f"fold{fm['fold']}_def_rmse": fm["def_rmse"],
                f"fold{fm['fold']}_em_rmse":  fm["em_rmse"],
                f"fold{fm['fold']}_off_r2":   fm["off_r2"],
                f"fold{fm['fold']}_def_r2":   fm["def_r2"],
            })
        if "fold3_em_rmse" in cv_results:
            mlflow.log_metric(GATE_METRIC, cv_results["fold3_em_rmse"])
        if "mean_em_rmse" in cv_results:
            mlflow.log_metric("mean_em_rmse", cv_results["mean_em_rmse"])

        # Residual stds
        mlflow.log_metric("off_resid_std", models.off_resid_std)
        mlflow.log_metric("def_resid_std", models.def_resid_std)

        # Slot baselines JSON (convert tuple keys to strings for JSON serialization)
        sb_json = {f"{k[0]}_{k[1]}": v for k, v in slot_baselines.items()}
        mlflow.log_dict(sb_json, "slot_baselines.json")

        # Coefficient report
        coef_report = {
            f: {"off_coef": float(models.off_model.coef_[i]),
                "def_coef": float(models.def_model.coef_[i])}
            for i, f in enumerate(ROSTER_FEATURES)
        }
        mlflow.log_dict(coef_report, "model_coefficients.json")

        # Serialize Ridge + scaler pickles as artifacts
        with tempfile.TemporaryDirectory() as tmpdir:
            for name, obj in [("off_model", models.off_model),
                              ("off_scaler", models.off_scaler),
                              ("def_model", models.def_model),
                              ("def_scaler", models.def_scaler)]:
                path = os.path.join(tmpdir, f"{name}.pkl")
                with open(path, "wb") as f:
                    pickle.dump(obj, f)
                mlflow.log_artifact(path, artifact_path=ARTIFACT_PATH)

        run_id = run.info.run_id

    # Promotion gate
    if "fold3_em_rmse" in cv_results:
        outcome = maybe_promote(
            client,
            MLFLOW_MODEL_NAME,
            run_id,
            ARTIFACT_PATH,
            GATE_METRIC,
            cv_results["fold3_em_rmse"],
            higher_is_better=False,   # lower RMSE = better
        )
        print(f"Run ID   : {run_id}")
        print(f"Registry : {outcome}")
    else:
        print(f"Run ID: {run_id} — no gate metric (CV was skipped)")
else:
    print("SHOULD_LOG_MLFLOW is False — skipping MLflow run.")
    print("Use scripts/run_team_rating_projection.py for production runs.")

## 8. Load 2027 Inference Data + Build School Baselines

Loads all tables needed for 2027 inference and builds one baseline roster feature vector
per school (returning players + slot fills). Hard gate: `playing_time_projections` for
`season=2027` must exist (from `scripts/run_playing_time.py`).

In [ ]:
# Cell 8 — Load inference data + build school baselines
print(f"Loading inference data for target_season={TARGET_SEASON}, source_season={SOURCE_SEASON}...")
data = trp.load_inference_data(engine, TARGET_SEASON, SOURCE_SEASON)

print(f"  school_meta     : {len(data['school_meta']):,} schools")
print(f"  baseline_members: {len(data['baseline_members']):,} player-school rows")
print(f"  prior_stats     : {len(data['prior_stats']):,} player-school rows")
print(f"  portal_ids      : {len(data['portal_ids']):,} candidates")
print(f"  neutral_proj    : {len(data['neutral_proj']):,} rows")
print(f"  playing_time    : {len(data['playing_time']):,} rows")
print(f"  he_rapm         : {len(data['he_rapm']):,} returner RAPM rows")

# Hard gate
if data['playing_time'].empty:
    raise RuntimeError(
        f"Hard gate: playing_time_projections is empty for season={TARGET_SEASON}.\n"
        f"Run: uv run python scripts/run_playing_time.py --target-season {TARGET_SEASON}"
    )

# Build school adj_em lookup
sm = data["school_meta"]
season_adj_ems = sm["adj_em"].dropna().to_numpy(dtype=float)
school_adj_ems = {int(k): float(v) for k, v in
                  sm.set_index("school_id")["adj_em"].dropna().to_dict().items()}

# Build baseline rosters
print(f"\nBuilding 2027 baseline rosters...")
school_baselines = trp.build_school_baselines(
    data, slot_baselines, school_adj_ems, season_adj_ems, TARGET_SEASON
)
print(f"  Schools with baseline rosters: {len(school_baselines):,}")

# Distribution of baseline adj_em (school strength)
baseline_ems = [v["adj_em"] for v in school_baselines.values()]
print(f"  Baseline adj_em: mean={np.mean(baseline_ems):.1f}  std={np.std(baseline_ems):.1f}"
      f"  min={np.min(baseline_ems):.1f}  max={np.max(baseline_ems):.1f}")

tier_counts = {t: sum(1 for v in school_baselines.values() if v["tier"] == t) for t in [1,2,3,4]}
print(f"  Conference tiers: {tier_counts}")

## 9. Sample Candidate Counterfactuals

Runs the full counterfactual pipeline for a sample of candidate × school pairs.
Change `SAMPLE_PLAYER_IDS` and `SAMPLE_SCHOOL_IDS` to inspect specific matchups.

For the full inference pass (~456K pairs), use `scripts/run_team_rating_projection.py`.
Set `RUN_FULL_INFERENCE = True` here only if you need to manually trigger it interactively.

In [ ]:
# Cell 9 — Sample candidate counterfactuals
RUN_FULL_INFERENCE = False   # True = all portal candidates × all schools; use the script instead
SAMPLE_N_PLAYERS = 10        # how many portal candidates to spot-check
SAMPLE_N_SCHOOLS = 20        # how many schools to spot-check per candidate

pt_df = data["playing_time"]
neutral_df_inf = data["neutral_proj"]

neutral_index: dict = {}
if not neutral_df_inf.empty:
    neutral_index = neutral_df_inf.set_index("player_id").to_dict("index")

pt_index: dict = {}
for _, row in pt_df.iterrows():
    pt_index[(int(row["player_id"]), int(row["school_id"]))] = row.to_dict()

# Determine inference scope
if RUN_FULL_INFERENCE:
    player_scope = data["portal_ids"]
    school_scope = list(school_baselines.keys())
    print(f"Full inference: {len(player_scope):,} players × {len(school_scope):,} schools")
else:
    player_scope = data["portal_ids"][:SAMPLE_N_PLAYERS]
    school_scope = list(school_baselines.keys())[:SAMPLE_N_SCHOOLS]
    print(f"Sample: {len(player_scope)} players × {len(school_scope)} schools")

records: list[dict] = []
n_skipped = 0

for player_id in player_scope:
    proj_row = neutral_index.get(int(player_id), {})
    cand_value = float(proj_row.get("value_per_100", 0.0))
    cand_proj = pd.Series({"value_per_100": cand_value})

    for school_id in school_scope:
        pt_key = (int(player_id), int(school_id))
        if pt_key not in pt_index:
            n_skipped += 1
            continue

        pt_row = pd.Series(pt_index[pt_key])
        baseline_info = school_baselines[school_id]

        candidate_rows, returning_pct = trp.build_candidate_roster(
            baseline_info, pt_row, cand_proj, slot_baselines
        )
        ca_features = trp.build_roster_features(
            candidate_rows, baseline_info["tier"],
            baseline_info["adj_tempo"], returning_pct, slot_baselines
        )
        delta = trp.compute_counterfactual(baseline_info["features"], ca_features, models)
        ci_lo, ci_hi = trp.build_confidence_interval(
            baseline_info["roster_rows"], candidate_rows, pt_row,
            models, slot_baselines, baseline_info["tier"],
            baseline_info["adj_tempo"], returning_pct,
        )

        records.append({
            "player_id":              int(player_id),
            "school_id":              school_id,
            "season":                 TARGET_SEASON,
            "current_adj_em":         delta["baseline_adj_em"],
            "projected_adj_em":       delta["projected_adj_em"],
            "delta_adj_em":           delta["delta_adj_em"],
            "baseline_adj_o":         delta["baseline_adj_o"],
            "baseline_adj_d":         delta["baseline_adj_d"],
            "projected_adj_o":        delta["projected_adj_o"],
            "projected_adj_d":        delta["projected_adj_d"],
            "ci_lower":               ci_lo,
            "ci_upper":               ci_hi,
            "expected_minutes_input": float(pt_row.get("expected_minutes", 0)),
            "candidate_usage_role":   str(pt_row.get("usage_role", "rotation")),
            "baseline_adj_em":        delta["baseline_adj_em"],
        })

print(f"Computed: {len(records):,} counterfactuals  Skipped (no PT row): {n_skipped:,}")

if records:
    results_df = pd.DataFrame(records)
    print("\ndelta_adj_em summary:")
    print(results_df["delta_adj_em"].describe().round(3))

    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, col in zip(axes, ["delta_adj_em", "delta_adj_o", "delta_adj_d"]):
        if col in results_df.columns:
            results_df[col].plot(kind="hist", bins=30, ax=ax, title=col)
        else:
            ax.set_title(f"{col} (not in results)")
    plt.tight_layout()
    plt.show()
else:
    results_df = pd.DataFrame()
    print("No counterfactuals computed — check PT data and portal_ids")

## 10. Explanation Payloads + CI Inspection

Builds explanation payloads (Ridge coef × feature delta by group) for the sample records
and inspects the CI width distribution — a wide CI means high RAPM uncertainty for that
player/school combination.

In [ ]:
# Cell 10 — Explanation payloads + CI inspection (sample)
if not results_df.empty:
    # Add explanations to sample records
    explanation_rows = []
    for _, rec in results_df.iterrows():
        player_id = int(rec["player_id"])
        school_id = int(rec["school_id"])
        pt_key = (player_id, school_id)
        if pt_key not in pt_index:
            continue
        pt_row = pd.Series(pt_index[pt_key])
        baseline_info = school_baselines[school_id]
        proj_row = neutral_index.get(player_id, {})
        cand_proj = pd.Series({"value_per_100": float(proj_row.get("value_per_100", 0.0))})

        candidate_rows, returning_pct = trp.build_candidate_roster(
            baseline_info, pt_row, cand_proj, slot_baselines
        )
        ca_features = trp.build_roster_features(
            candidate_rows, baseline_info["tier"],
            baseline_info["adj_tempo"], returning_pct, slot_baselines
        )
        delta = {
            "delta_adj_o":  rec.get("delta_adj_o", 0.0),
            "delta_adj_d":  rec.get("delta_adj_d", 0.0),
            "delta_adj_em": rec["delta_adj_em"],
            "baseline_adj_o": rec["baseline_adj_o"],
            "baseline_adj_d": rec["baseline_adj_d"],
        }
        expl = trp.build_explanation_payload(baseline_info["features"], ca_features, models, pt_row, delta)
        expl["player_id"] = player_id
        expl["school_id"] = school_id
        explanation_rows.append(expl)

    if explanation_rows:
        expl_df = pd.DataFrame(explanation_rows)
        print("Explanation component summary:")
        expl_cols = [c for c in expl_df.columns if c.endswith("_delta") or c.endswith("_contribution")]
        display(expl_df[expl_cols].describe().round(3))

        # Stacked attribution bar for top-10 by delta_adj_em
        top10 = results_df.nlargest(10, "delta_adj_em").merge(
            expl_df[["player_id", "school_id"] + expl_cols], on=["player_id", "school_id"], how="left"
        )
        if not top10.empty and expl_cols:
            fig, ax = plt.subplots(figsize=(12, 4))
            bottom = np.zeros(len(top10))
            colors = plt.cm.tab10.colors
            for i, col in enumerate(expl_cols[:6]):
                if col in top10.columns:
                    vals = top10[col].fillna(0).values
                    ax.bar(range(len(top10)), np.maximum(vals, 0), bottom=np.maximum(bottom, 0),
                           color=colors[i], label=col.replace("_delta", ""), alpha=0.8)
                    ax.bar(range(len(top10)), np.minimum(vals, 0), bottom=np.minimum(bottom, 0),
                           color=colors[i], alpha=0.8)
                    bottom = bottom + vals
            ax.axhline(0, color="black", linewidth=0.8)
            ax.set_xticks(range(len(top10)))
            ax.set_xticklabels(
                [f"p{str(row['player_id'])[-4:]}→s{row['school_id']}" for _, row in top10.iterrows()],
                rotation=45, fontsize=7
            )
            ax.set_ylabel("Component contribution to delta_adj_em")
            ax.set_title("Attribution decomposition — top-10 delta_adj_em pairs")
            ax.legend(loc="upper right", fontsize=7)
            plt.tight_layout()
            plt.show()

    # CI width distribution
    if "ci_lower" in results_df.columns and "ci_upper" in results_df.columns:
        results_df["ci_width"] = results_df["ci_upper"] - results_df["ci_lower"]
        print("\n80% CI width summary:")
        print(results_df["ci_width"].describe().round(3))

        fig, ax = plt.subplots(figsize=(6, 3))
        results_df["ci_width"].plot(kind="hist", bins=30, ax=ax,
                                     title="80% CI width distribution (delta_adj_em)")
        ax.set_xlabel("CI width (AdjEM points)")
        plt.tight_layout()
        plt.show()
else:
    print("No records from Cell 9 — run Cell 9 first")

## 11. DB Write + Percentile Computation (Gated)

Adds national percentile and conference rank to records, then upserts to
`team_rating_projections`. Set `SHOULD_WRITE = True` only after Cell 9 has run in
full-inference mode or after using `scripts/run_team_rating_projection.py`.

**Production writes belong in the script.** This cell is a one-off safety valve.

In [ ]:
# Cell 11 — Percentiles + DB write (gated)
SHOULD_WRITE = False   # flip to True only for one-off writes

if SHOULD_WRITE and records:
    print(f"Adding national percentiles to {len(records):,} records...")
    records_with_pct = trp.compute_national_percentiles(records, data["school_meta"])

    pct_vals = [r["national_percentile"] for r in records_with_pct]
    print(f"  national_percentile: mean={np.mean(pct_vals):.1f}  min={np.min(pct_vals)}  max={np.max(pct_vals)}")

    print(f"\nUpserting {len(records_with_pct):,} rows to team_rating_projections...")
    n_written = trp.upsert_team_rating_projections(engine, records_with_pct, MODEL_VERSION)
    print(f"  Done: {n_written:,} rows written (model_version={MODEL_VERSION})")
elif SHOULD_WRITE and not records:
    print("SHOULD_WRITE is True but no records to write — run Cell 9 first")
else:
    print("SHOULD_WRITE is False — no DB write performed.")
    print("Use scripts/run_team_rating_projection.py for production runs.")

## 12. Spot-Checks

Diagnostic queries against the actual `team_rating_projections` table (reads from DB,
independent of what was computed in Cells 9-11). Useful after a script run to validate
the written rows.

Change `SPOT_SCHOOL_ID` to any school in your DB.

In [ ]:
# Cell 12 — Spot-checks on written rows
SPOT_SCHOOL_ID = 1    # change to any school_id in your DB

with engine.connect() as conn:
    # Top-20 delta_adj_em pairs for TARGET_SEASON
    top_pairs = conn.execute(text("""
        SELECT trp.player_id, trp.school_id, s.name AS school_name,
               trp.delta_adj_em, trp.baseline_adj_o, trp.baseline_adj_d,
               trp.projected_adj_o, trp.projected_adj_d,
               trp.ci_lower, trp.ci_upper, trp.expected_minutes_input,
               trp.candidate_usage_role, trp.national_percentile, trp.conference_rank
        FROM team_rating_projections trp
        JOIN schools s ON s.id = trp.school_id
        WHERE trp.season = :season AND trp.model_version = :version
        ORDER BY trp.delta_adj_em DESC
        LIMIT 20
    """), {"season": TARGET_SEASON, "version": MODEL_VERSION}).fetchall()

    # Top candidates for one school
    school_rows = conn.execute(text("""
        SELECT trp.player_id, trp.delta_adj_em, trp.delta_adj_o, trp.delta_adj_d,
               trp.ci_lower, trp.ci_upper, trp.expected_minutes_input, trp.candidate_usage_role,
               trp.national_percentile, trp.conference_rank
        FROM team_rating_projections trp
        WHERE trp.season = :season
          AND trp.school_id = :school_id
          AND trp.model_version = :version
        ORDER BY trp.delta_adj_em DESC
        LIMIT 15
    """), {"season": TARGET_SEASON, "school_id": SPOT_SCHOOL_ID, "version": MODEL_VERSION}).fetchall()

    # Summary stats from DB
    summary_row = conn.execute(text("""
        SELECT count(*) AS n_rows,
               count(DISTINCT player_id) AS n_players,
               count(DISTINCT school_id) AS n_schools,
               avg(delta_adj_em) AS mean_delta,
               stddev(delta_adj_em) AS std_delta,
               min(delta_adj_em) AS min_delta,
               max(delta_adj_em) AS max_delta
        FROM team_rating_projections
        WHERE season = :season AND model_version = :version
    """), {"season": TARGET_SEASON, "version": MODEL_VERSION}).fetchone()

if summary_row and summary_row[0] and summary_row[0] > 0:
    n, np_, ns, mu, sd, mn, mx = summary_row
    print(f"--- Summary (season={TARGET_SEASON}, model_version={MODEL_VERSION}) ---")
    print(f"  rows={n:,}  players={np_:,}  schools={ns:,}")
    print(f"  delta_adj_em: mean={mu:.3f}  std={sd:.3f}  min={mn:.3f}  max={mx:.3f}")

    if top_pairs:
        print(f"\n--- Top-20 delta_adj_em pairs (all schools, season={TARGET_SEASON}) ---")
        cols = ["player_id", "school_name", "delta_adj_em", "ci_lower", "ci_upper",
                "expected_minutes_input", "candidate_usage_role", "national_percentile"]
        display(pd.DataFrame(top_pairs, columns=[
            "player_id", "school_id", "school_name",
            "delta_adj_em", "baseline_adj_o", "baseline_adj_d",
            "projected_adj_o", "projected_adj_d",
            "ci_lower", "ci_upper", "expected_minutes_input",
            "candidate_usage_role", "national_percentile", "conference_rank"
        ])[["player_id", "school_name", "delta_adj_em", "ci_lower", "ci_upper",
            "expected_minutes_input", "candidate_usage_role", "national_percentile"]].round(3))

    if school_rows:
        print(f"\n--- Top-15 candidates for school_id={SPOT_SCHOOL_ID}, season={TARGET_SEASON} ---")
        display(pd.DataFrame(school_rows, columns=[
            "player_id", "delta_adj_em", "delta_adj_o", "delta_adj_d",
            "ci_lower", "ci_upper", "expected_minutes_input", "candidate_usage_role",
            "national_percentile", "conference_rank"
        ]).round(3))
    else:
        print(f"No rows for school_id={SPOT_SCHOOL_ID} — check that this school_id exists")
else:
    print(f"No rows in team_rating_projections for season={TARGET_SEASON} model_version={MODEL_VERSION}")
    print("Run scripts/run_team_rating_projection.py or set SHOULD_WRITE=True in Cell 11 first.")